In [6]:
import os
import json
import zipfile
import numpy as np
import copy
import requests

import pandas as pd
import geopandas as gpd

import shapely.wkt as wkt
from shapely.geometry import LineString
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import BoundaryNorm

from math import radians, cos, sin, asin, sqrt

In [ ]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"Chicago")
os.makedirs(DIR, exist_ok=True)

Chicago_Data_Portal_KEY_ID = ""
Chicago_Data_Portal_KEY_SECRET = ""

chicago_shape="./Chicago Community Areas.geojson"
chi_shapefile_url="https://data.cityofchicago.org/api/v3/views/igwz-8jzy/query.geojson"

taxi_orig_file=os.path.join(DIR,"chi_taxi_2025_15min.csv")
TNP_orig_file=os.path.join(DIR,"chi_TNP_2025_15min.csv")
bike_orig_file=os.path.join(DIR,"chi_bike_2025_15min.csv")
scooter_orig_file=os.path.join(DIR,"chi_scooter_2025_60min.csv")

# Download
Community Areas https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-Map/cauq-8yn6  
Taxi 2025 15min https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data    
Transportation Network Providers 2025 15min https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2025-/6dvr-xwnh/about_data  
Bike 2025 15min https://data.cityofchicago.org/Transportation/Divvy-Trips/fg6s-gzvg/about_data  
Scooter 2025 1hour https://data.cityofchicago.org/Transportation/E-Scooter-Trips/2i5w-ykuw/about_data  

## 1 Shapefile

In [8]:
if not os.path.exists(chicago_shape):
    response = requests.get(chi_shapefile_url)
    with open(chicago_shape, "w", encoding="utf-8") as file:
        file.write(response.text)

## 2 Taxi

In [9]:
def download_from_api(API_URL, query, file):
    response = requests.get(
        API_URL, 
        auth=(Chicago_Data_Portal_KEY_ID, Chicago_Data_Portal_KEY_SECRET), 
        params=query,
        stream=True,
        timeout=120
    )
    if response.status_code == 200:
        lines = response.iter_lines(decode_unicode=True)
        with open(file, mode="w", encoding="utf-8") as f:
            line_count = 0
            for line in lines:
                if line:
                    f.write(line + "\n")
                    line_count += 1
                    if line_count % 200000 == 0:
                        print(f"Downloaded {line_count} ...")
        print(f"\nSaved {line_count}")
    else:
        print(f"Error: {response.status_code}")
        print(f"Info: {response.text}")

In [ ]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/api/v3/views/b8xg-w8bx/query.csv"
    query_params = {
        "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' AND trip_start_timestamp < '2026-01-01T00:00:00'",
        "$limit": 20000000, 
        "$order": "trip_start_timestamp ASC"
    }
    download_from_api(API_URL, query_params, taxi_orig_file)

Downloaded 200000 ...
Downloaded 400000 ...
Downloaded 600000 ...
Downloaded 800000 ...
Downloaded 1000000 ...
Downloaded 1200000 ...
Downloaded 1400000 ...
Downloaded 1600000 ...
Downloaded 1800000 ...
Downloaded 2000000 ...
Downloaded 2200000 ...
Downloaded 2400000 ...
Downloaded 2600000 ...
Downloaded 2800000 ...
Downloaded 3000000 ...
Downloaded 3200000 ...
Downloaded 3400000 ...
Downloaded 3600000 ...
Downloaded 3800000 ...
Downloaded 4000000 ...
Downloaded 4200000 ...
Downloaded 4400000 ...
Downloaded 4600000 ...
Downloaded 4800000 ...
Downloaded 5000000 ...
Downloaded 5200000 ...
Downloaded 5400000 ...
Downloaded 5600000 ...
Downloaded 5800000 ...
Downloaded 6000000 ...
Downloaded 6200000 ...
Downloaded 6400000 ...
Downloaded 6600000 ...
Downloaded 6800000 ...

Saved 6825839


## 3 Bike

In [11]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/api/v3/views/fg6s-gzvg/query.csv"
    query_params = {
        "$where": "start_time >= '2025-01-01T00:00:00' AND start_time < '2026-01-01T00:00:00'",
        "$limit": 20000000, 
        "$order": "start_time ASC"
    }
    download_from_api(API_URL, query_params, bike_orig_file)

Downloaded 200000 ...
Downloaded 400000 ...
Downloaded 600000 ...
Downloaded 800000 ...
Downloaded 1000000 ...
Downloaded 1200000 ...
Downloaded 1400000 ...
Downloaded 1600000 ...
Downloaded 1800000 ...
Downloaded 2000000 ...
Downloaded 2200000 ...
Downloaded 2400000 ...
Downloaded 2600000 ...
Downloaded 2800000 ...
Downloaded 3000000 ...
Downloaded 3200000 ...
Downloaded 3400000 ...
Downloaded 3600000 ...
Downloaded 3800000 ...
Downloaded 4000000 ...
Downloaded 4200000 ...
Downloaded 4400000 ...
Downloaded 4600000 ...
Downloaded 4800000 ...
Downloaded 5000000 ...
Downloaded 5200000 ...
Downloaded 5400000 ...
Downloaded 5600000 ...
Downloaded 5800000 ...
Downloaded 6000000 ...
Downloaded 6200000 ...
Downloaded 6400000 ...
Downloaded 6600000 ...
Downloaded 6800000 ...
Downloaded 7000000 ...
Downloaded 7200000 ...
Downloaded 7400000 ...
Downloaded 7600000 ...
Downloaded 7800000 ...
Downloaded 8000000 ...
Downloaded 8200000 ...
Downloaded 8400000 ...
Downloaded 8600000 ...
Downloaded 8800

## 4 Scooter

In [ ]:
DOWNLOAD=True
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/api/v3/views/2i5w-ykuw/query.csv"
    query_params = {
        "$where": "start_time >= '2025-01-01T00:00:00' AND start_time < '2026-01-01T00:00:00'",
        "$limit": 20000000, 
        "$order": "start_time ASC"
    }
    download_from_api(API_URL, query_params, scooter_orig_file)

Downloaded 200000 ...
Downloaded 400000 ...
Downloaded 600000 ...
Downloaded 800000 ...
Downloaded 1000000 ...
Downloaded 1200000 ...


## 5 TNP

In [ ]:
DOWNLOAD=True
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/api/v3/views/6dvr-xwnh/query.csv"
    query_params = {
        "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' AND trip_start_timestamp < '2026-01-01T00:00:00'",
        "$limit": 200000000, 
        "$order": "trip_start_timestamp ASC"
    }
    download_from_api(API_URL, query_params, TNP_orig_file)